# Sessao Spark para uso do delta sharing

## Usando spark-submit
Se você estiver executando seu código como um aplicativo com spark-submit, você pode incluir o pacote usando a flag --packages:

```shell
spark-submit --packages io.delta:delta-sharing-spark_2.12:0.7.0 seu_script.py
```

## Usando pyspark ou Jupyter/Databricks
Se você estiver usando pyspark interativamente ou em um notebook (como Jupyter), você pode configurar isso ao criar sua SparkSession:

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DeltaSharingApp") \
    .config("spark.jars.packages", "io.delta:delta-sharing-spark_2.12:0.7.0") \
    .getOrCreate()

df = spark.read.format("deltaSharing").load(table_url)
```

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import delta_sharing
import logging

In [2]:
spark = (
    SparkSession
    .builder.appName("DeltaSharingApp")
    .config("spark.jars.packages",
            "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1,"
            "io.delta:delta-sharing-spark_2.12:0.7.0")
    .getOrCreate()
)

In [3]:
profile_file = "./config.share"
client = delta_sharing.SharingClient(profile_file)

In [4]:
client.list_all_tables()

[Table(name='contact', share='core_share', schema='deltashare_core'),
 Table(name='broad_notifications_dataflow', share='core_share', schema='deltashare_core'),
 Table(name='messages', share='core_share', schema='deltashare_core'),
 Table(name='notifications', share='core_share', schema='deltashare_core'),
 Table(name='tickets_new', share='core_share', schema='deltashare_core'),
 Table(name='eventtracks', share='core_share', schema='deltashare_core')]

In [5]:
share_name = "core_share"
schema_name = "deltashare_core"

In [6]:
#table_url = profile_file + "#<share-name>.<schema-name>.<table-name>"
table_url = profile_file + f"#{share_name}.{schema_name}.messages"

In [7]:
df_streaming = (
    spark.readStream
    .format("deltaSharing")
    .option("ignoreDeletes", "true")
    .load(table_url)
)

logging.info("✅ Streaming iniciado com sucesso")

# df_streaming = df_streaming.dropDuplicates()

In [ ]:
if 'StorageDateDayBR' in df_streaming.columns:
    query = (
        df_streaming.coalesce(2)
        .writeStream
        .outputMode("append")
        .format("json")
        .option("path", "streaming/data")
        # .option("path", "streaming/data_2_dropDuplicate")
        .option("maxRecordsPerFile", 1000)
        .partitionBy('StorageDateDayBR')
        .trigger(processingTime="5 minutes")
        .option("checkpointLocation", "streaming/checkpoint")
        # .option("checkpointLocation", "streaming/checkpoint_2_dropDuplicate")
        .start()
    )
    logging.info("Escrevendo dados em: streaming/data")

    query.awaitTermination()